In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import glob
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import plotly.graph_objs as go
import gzip
import h5py
import scanpy as sc
import scipy
import anndata as ad
import gseapy as gp
from matplotlib import cm
from matplotlib.colors import to_hex

In [ ]:
# Set rcParams for a white background globally
plt.rcParams['figure.facecolor'] = 'white'  # Set figure background
plt.rcParams['axes.facecolor'] = 'white'    # Set axes background
plt.rcParams['savefig.facecolor'] = 'white' # For saving files with white background
# Set rcParams for black axes and ticks globally
plt.rcParams['axes.edgecolor'] = 'black'  # Axis lines
plt.rcParams['xtick.color'] = 'black'     # X-axis ticks
plt.rcParams['ytick.color'] = 'black'     # Y-axis ticks
# Set TrueType fonts for PDF output
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
adata = sc.read_h5ad("/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/h5_files/merged_post_mira.h5ad")

In [ ]:
adata.obs['ko_status'] = adata.obs['group'].apply(
    lambda x: 'ko' if 'ko' in x else ('ntc' if 'ntc' in x else x))

In [ ]:
print("X_topic_compositions shape:", adata.obsm['X_topic_compositions'].shape)
print("X_umap shape:", adata.obsm['X_umap'].shape)
print("X_umap_features shape:", adata.obsm['X_umap_features'].shape)

# DEG analysis

In [ ]:
# Loop through each unique Leiden cluster and run DE analysis
for cluster in adata.obs['leiden'].unique():
    # Subset to the current cluster
    adata_cluster = adata[adata.obs['leiden'] == cluster].copy()

    # Further subset to the two groups of interest
    adata_subset = adata_cluster[adata_cluster.obs['ko_status'].isin(['ko', 'ntc'])].copy()

    # Skip if there are not enough cells in both groups
    if adata_subset.obs['ko_status'].nunique() < 2:
        print(f"Skipping cluster {cluster}: not enough groups")
        continue

    # Run differential expression
    key_added = f'deg_{cluster}'
    sc.tl.rank_genes_groups(
        adata_subset,
        'ko_status',
        key_added=key_added,
        method='t-test_overestim_var',
        use_raw=False,
        pts=True
    )

    # Export results
    df_de_gene = sc.get.rank_genes_groups_df(adata_subset, group=None, key=key_added)
    filename = f'/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/tables/bcl6ko_cluster{cluster}.csv'
    df_de_gene.to_csv(filename, header=True, index=False)

# gene set enrichment

In [ ]:
#import gene lists
path = '/ix/djishnu/peasena/gene_sets/human_tonsil_atlas_GC_vs_ActB_cleaned.csv'

tonsil_GC_actB = pd.read_csv(path)
tonsil_GC_actB.columns.values[2] = "logfoldchanges"
tonsil_GC_actB

In [ ]:
#import G1 preGC cluster BCL6 KO deg (cluster 3)
path = '/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/tables/bcl6ko_cluster3.csv'

g1_pregc_bcl6ko = pd.read_csv(path)

#filter group = ko
g1_pregc_bcl6ko = g1_pregc_bcl6ko[g1_pregc_bcl6ko['group'] == 'ko'].copy()

g1_pregc_bcl6ko

In [ ]:

def get_top_genes(df, gene_col='names', score_col='scores', top_n=150):
    up_df = df.loc[df[score_col] > 0].copy()
    up_df = up_df.sort_values(by=score_col, ascending=False)
    return up_df[gene_col].astype(str).tolist()[:top_n]

tonsil_GC_actB_set    = get_top_genes(tonsil_GC_actB)

gene_sets = {
    'tonsil_GC_actB': tonsil_GC_actB_set
}

In [ ]:
# prep the ranked list from in vitro preGC BCL6 KO (cluster 3)
rnk_df = (
    g1_pregc_bcl6ko[['names', 'scores']]
    .dropna()
    .rename(columns={'scores': 'score', 'names': 'gene'})
)


In [ ]:
# 3) GSEAPY prerank
pre_res = gp.prerank(
    rnk=rnk_df,
    gene_sets=gene_sets,
    threads=4,
    permutation_num=1000,
    seed=42,
    outdir=None,
    min_size=10,
    max_size=5000
)

pre_res.res2d

In [ ]:
pre_res.res2d['Lead_genes'][1]

In [ ]:
sc.set_figure_params(dpi=300, fontsize=15)
sns.set(rc={'figure.figsize':(4,4)})

In [ ]:
terms = pre_res.res2d['Term']
axs = pre_res.plot(terms='tonsil_GC_actB')  # This draws the plot

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

def overlay_enrichment_with_style(
    pre_res,
    terms,
    color_map,
    figsize=(6, 5),
    rug_frac=0.20,                 # fraction of rug row height each tick spans
    es_rug_height_ratio=(1.0, 0.30)  # ES panel taller than rug panel
):
    """
    Overlay ES curves + rugs for specified terms with custom colors and styling.
    Returns: (fig, (ax_es, ax_rug))
    - pre_res: gseapy.prerank result object
    - terms: list[str] of set names present in pre_res.res2d['Term']
    - color_map: dict {term: color_hex}
    - rug_frac: shorter rugs if < 1 (e.g., 0.2)
    - es_rug_height_ratio: (ES panel height, rug panel height)
    """
    # Prepare figure up-front so we can always return something
    fig = plt.figure(figsize=figsize)
    fig.patch.set_facecolor('white')

    try:
        # Validate terms
        available = set(pre_res.res2d["Term"])
        terms = [t for t in terms if t in available]
        if len(terms) == 0:
            print("[WARN] None of the requested terms are present in pre_res.res2d['Term'].")
            gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=es_rug_height_ratio)
            ax_es = fig.add_subplot(gs[0])
            ax_rug = fig.add_subplot(gs[1], sharex=ax_es)
            ax_es.text(0.5, 0.5, "No matching terms", ha='center', va='center')
            ax_es.set_axis_off()
            ax_rug.set_axis_off()
            plt.tight_layout()
            return fig, (ax_es, ax_rug)

        n_terms = len(terms)
        n_genes = len(pre_res.ranking)

        # Layout
        gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=es_rug_height_ratio)
        ax_es = fig.add_subplot(gs[0])
        ax_rug = fig.add_subplot(gs[1], sharex=ax_es)

        # Styling: white background, black spines/ticks
        for ax in (ax_es, ax_rug):
            ax.set_facecolor('white')
            for spine in ax.spines.values():
                spine.set_edgecolor('black')
                spine.set_linewidth(1.2)
            ax.tick_params(colors='black', labelcolor='black')

        # Hide x on ES; only rug shows x-axis label
        ax_es.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
        ax_es.set_xlabel("")

        # Plot each term
        for i, term in enumerate(terms):
            res = pre_res.results.get(term, {})
            # Handle key names across gseapy versions
            es_profile = res.get('es_profile', res.get('RES', None))
            hit_ind    = res.get('hit_ind',    res.get('hits', None))

            if es_profile is None:
                print(f"[WARN] Term '{term}' lacks ES profile ('es_profile'/'RES'). Using zeros.")
                es_profile = np.zeros(n_genes)
            else:
                es_profile = np.array(es_profile)

            if hit_ind is None:
                print(f"[WARN] Term '{term}' lacks hit indices ('hit_ind'/'hits').")
                hit_ind = np.array([], dtype=int)
            else:
                hit_ind = np.array(hit_ind, dtype=int)

            color = color_map.get(term, None)

            # ES curve
            ax_es.plot(np.arange(n_genes), es_profile, lw=2.2, color=color, label=term)

            # Rugs: place each term in its own row; shorter ticks via `rug_frac`
            y0 = i / max(n_terms, 1)
            y1 = y0 + (rug_frac / max(n_terms, 1))
            for x in hit_ind:
                ax_rug.vlines(x, y0, y1, colors=color, lw=1)

        # Decorations
        ax_es.axhline(0.0, color='black', lw=1, ls='--')
        ax_es.set_ylabel("Enrichment Score", color='black')
        ax_es.set_xlim(0, n_genes - 1)
        ax_es.legend(loc='center left', bbox_to_anchor=(1.02, 0.05), frameon=False)

        ax_rug.set_xlabel("Gene Rank", color='black')
        ax_rug.set_yticks([])
        ax_rug.set_ylim(0, 1)

        plt.tight_layout()
        return fig, (ax_es, ax_rug)

    except Exception as e:
        # Ensure we always return a figure/axes even on error
        print(f"[ERROR] overlay_enrichment_with_style failed: {e}")
        gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=es_rug_height_ratio)
        ax_es = fig.add_subplot(gs[0])
        ax_rug = fig.add_subplot(gs[1], sharex=ax_es)
        ax_es.text(0.5, 0.5, f"Plot failed:\n{e}", ha='center', va='center', color='red')
        ax_rug.set_axis_off()
        plt.tight_layout()
       



In [ ]:
terms = ["tonsil_GC_actB"]
color_map = {
    "tonsil_GC_actB": "#1b9e77"
}

fig, axs = overlay_enrichment_with_style(
    pre_res,
    terms,
    color_map,
    figsize=(6, 4),
    rug_frac=1,                   
    es_rug_height_ratio=(1, 0.125)
)

# Save
fig.savefig(
    "/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/plots/pb_gc_bcl6ko_deg_enrichment.pdf",
    dpi=300,
    bbox_inches="tight"
)
